
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 4L: Optimizing Apache Spark

In this lab, you'll apply the Spark optimization techniques you've learned to a real-world dataset - the Airline Performance dataset.

### Objectives
- Analyze and understand dataset partitioning
- Implement proper partition strategies
- Use caching effectively to improve performance
- Analyze execution plans and measure performance improvements


## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Data Setup

Let's begin by loading the data and understanding our environment.

In [0]:
## Import necessary libraries
from pyspark.sql.functions import col, count, avg, sum, max
import time

## Load the dataset
flights_df = spark.read.table("dbacademy_airline.v01.flights")

num_cores = sc.defaultParallelism
print(f"Default parallelism (cores): {num_cores}")

shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Default shuffle partitions: {shuffle_partitions}")

Default parallelism (cores): 4
Default shuffle partitions: 200


In [0]:
flights_df.printSchema()
display(flights_df.limit(5))

root
 |-- id: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- ArrDelay: string (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- TailNum: string (nullable = true)



id,year,FlightNum,ArrDelay,UniqueCarrier,TailNum
936302871625,2005,1,-4,AA,N328AA
936302871626,2005,1,38,AA,N376AA
936302871627,2005,1,4,AA,N335AA
936302871628,2005,1,-25,AA,N329AA
936302871629,2005,1,-5,AA,N319AA


## B. Understanding the Current Partitioning

Before applying optimizations, it's important to understand the current state of our data.

In [0]:
current_partitions = flights_df.rdd.getNumPartitions()
print(f"Current number of partitions: {current_partitions}")

Current number of partitions: 63


## C. Optimizing Partitions

Given skewed data, create an optimally distributed dataset.

In [0]:
# The following narrow transformation will introduce skew in the dataset
filtered_flights_df = flights_df.filter(
    (col("ArrDelay") > 30) & 
    col("UniqueCarrier").isin(["UA", "AA", "DL"]) &
    (col("year") == 2008)
)

In [0]:
filtered_flights_df = filtered_flights_df.repartition(10)
current_partitions = filtered_flights_df.rdd.getNumPartitions()
print(f"Current number of partitions: {current_partitions}")

Current number of partitions: 10


## D. Caching Strategies

Apply caching techniques and measure performance impact.  Optimize the following query.

In [0]:
print(f"There are a total of {filtered_flights_df.count()} filtered flights")

display(
    filtered_flights_df.groupBy("UniqueCarrier").agg(count("UniqueCarrier").alias("count"))
)

display(
    filtered_flights_df.groupBy("UniqueCarrier").agg(avg("ArrDelay").alias("avg_delay"))
)

There are a total of 2268215 filtered flights


UniqueCarrier,count
UA,738598
AA,1005708
DL,523909


UniqueCarrier,avg_delay
UA,85.4734727145213
AA,81.83265321544623
DL,79.00730661240787


In [0]:
filtered_flights_df.cache()

print(f"There are a total of {filtered_flights_df.count()} filtered flights")

display(
    filtered_flights_df.groupBy("UniqueCarrier").agg(count("UniqueCarrier").alias("count"))
)

display(
    filtered_flights_df.groupBy("year").agg(count("year").alias("count"))
)

There are a total of 2268215 filtered flights


UniqueCarrier,count
UA,738598
AA,1005708
DL,523909


year,count
2008,2268215


In [0]:
filtered_flights_df.unpersist()

DataFrame[id: bigint, year: int, FlightNum: int, ArrDelay: string, UniqueCarrier: string, TailNum: string]


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
